# Inference 

Формирует `predictions.csv` через baseline-checkpoint.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.feature_engineering import build_o2_feature_tables
from src.train_hierarchical_model import (
    HierarchicalScenarioRegressor,
    ScenarioHierarchicalDataset,
    make_collate_fn,
    predict_for_test,
)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
RUNTIME_DIR = ROOT / "_notebook_runtime_o2"
CHECKPOINT_PATH = ROOT / "model" / "hierarchical_o2_baseline.pt"
FINAL_PREDICTIONS_PATH = ROOT / "predictions.csv"

TRAIN_PATH = DATA_DIR / "daimler_mixtures_train.csv"
TEST_PATH = DATA_DIR / "daimler_mixtures_test.csv"
PROPS_PATH = DATA_DIR / "daimler_component_properties.csv"

for path in (TRAIN_PATH, TEST_PATH, PROPS_PATH, CHECKPOINT_PATH):
    if not path.exists():
        raise FileNotFoundError(path)


def load_checkpoint_feature_spec(raw_feature_spec: dict) -> dict:
    feature_spec = dict(raw_feature_spec)
    for key in (
        "component_numeric_mean",
        "component_numeric_std",
        "global_mean",
        "global_std",
        "target_mean",
        "target_std",
    ):
        feature_spec[key] = np.asarray(feature_spec[key], dtype=np.float32)
    return feature_spec


_, component_test_path, _, scenario_test_path = build_o2_feature_tables(
    TRAIN_PATH,
    TEST_PATH,
    PROPS_PATH,
    RUNTIME_DIR,
)
component_test_df = pd.read_csv(component_test_path)
scenario_test_df = pd.read_csv(scenario_test_path)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
feature_spec = load_checkpoint_feature_spec(checkpoint["feature_spec"])

categorical_cardinalities = {
    column: len(vocabulary)
    for column, vocabulary in feature_spec["category_vocabularies"].items()
}

model = HierarchicalScenarioRegressor(
    component_numeric_dim=len(feature_spec["component_numeric_mean"]),
    global_dim=len(feature_spec["global_columns"]),
    categorical_cardinalities=categorical_cardinalities,
    family_cardinality=len(feature_spec["family_vocabulary"]),
    model_dim=64,
    dropout=0.15,
)
model.load_state_dict(checkpoint["model_state_dict"])

test_dataset = ScenarioHierarchicalDataset(
    component_df=component_test_df,
    scenario_df=scenario_test_df,
    feature_spec=feature_spec,
    is_train=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=make_collate_fn(feature_spec, is_train=False),
)

predictions_df = predict_for_test(model, test_loader, torch.device("cpu"), feature_spec)
predictions_df.to_csv(FINAL_PREDICTIONS_PATH, index=False)
print(f"Final predictions saved to: {FINAL_PREDICTIONS_PATH.resolve()}")
